In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Case Study") \
    .master("local[*]") \
    .getOrCreate()




df = spark.read.csv(
    "data/orders.csv",
    header=True,
    inferSchema=True
)



In [4]:
customers = spark.read.csv(
    "data/customers.csv",
    header=True,
    inferSchema=True
)

orders = spark.read.csv(
    "data/orders.csv",
    header=True,
    inferSchema=True
)

order_items = spark.read.csv(
    "data/order_items.csv",
    header=True,
    inferSchema=True
)

products = spark.read.csv(
    "data/products.csv",
    header=True,
    inferSchema=True
)

returns = spark.read.csv(
    "data/returns.csv",
    header=True,
    inferSchema=True
)

In [7]:
customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")
products.createOrReplaceTempView("products")
returns.createOrReplaceTempView("returns")

In [8]:
spark.sql("""
SELECT
    (SELECT COUNT(DISTINCT customer_id) FROM customers) AS total_customers,
    (SELECT COUNT(DISTINCT product_id) FROM products) AS total_products,
    (SELECT COUNT(DISTINCT order_id) FROM orders) AS total_orders,
    (SELECT COUNT(DISTINCT order_id) FROM returns) AS total_returned_orders
""").show()

+---------------+--------------+------------+---------------------+
|total_customers|total_products|total_orders|total_returned_orders|
+---------------+--------------+------------+---------------------+
|         100000|         50000|     1000000|               100000|
+---------------+--------------+------------+---------------------+



In [14]:
spark.sql("""
SELECT
    p.category,
    SUM(oi.quantity * oi.selling_price) AS total_sales
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_sales DESC
""").show()

+--------------+-------------------+
|      category|        total_sales|
+--------------+-------------------+
|        Beauty|7.626693058999996E8|
|Home & Kitchen|7.581388732799999E8|
|         Books|7.464907783500013E8|
|          Toys|7.446190723000015E8|
|   Electronics|7.442665041099973E8|
|        Sports|7.433388681300002E8|
|      Clothing|7.419227945700002E8|
+--------------+-------------------+



In [15]:
spark.sql("""SELECT
    c.customer_id,
    c.customer_name,
    ROUND(SUM(oi.quantity * oi.selling_price), 2) AS total_purchase
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY
    c.customer_id,
    c.customer_name
ORDER BY total_purchase DESC
LIMIT 10;""").show()

+-----------+--------------+--------------+
|customer_id| customer_name|total_purchase|
+-----------+--------------+--------------+
|      93094|Customer_93094|     181569.68|
|      64560|Customer_64560|      169060.4|
|      23289|Customer_23289|      161573.8|
|      52275|Customer_52275|     153364.79|
|      61218|Customer_61218|     153067.55|
|      52034|Customer_52034|     152680.05|
|      40442|Customer_40442|     151037.32|
|      60528|Customer_60528|     148691.95|
|      84830|Customer_84830|     148363.84|
|      82593|Customer_82593|     148281.04|
+-----------+--------------+--------------+



In [16]:
spark.sql("""WITH latest_year AS (
    SELECT MAX(YEAR(order_date)) AS yr
    FROM orders
)

SELECT
    MONTH(o.order_date) AS month,
    ROUND(SUM(oi.quantity * oi.selling_price), 2) AS monthly_sales
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
CROSS JOIN latest_year
WHERE YEAR(o.order_date) = latest_year.yr
GROUP BY MONTH(o.order_date)
ORDER BY month;""").show()

+-----+--------------+
|month| monthly_sales|
+-----+--------------+
|    1|4.4457777576E8|
|    2| 4.153661442E8|
|    3|4.4362824541E8|
|    4|4.2782097434E8|
|    5|4.4481061895E8|
|    6|4.3170515406E8|
|    7|4.4367051912E8|
|    8|4.4109517702E8|
|    9|4.3107152608E8|
|   10|4.4136378931E8|
|   11|4.3362336404E8|
|   12|4.4271290835E8|
+-----+--------------+



In [17]:
spark.sql("""SELECT
    p.category,
    ROUND(
        COUNT(DISTINCT r.order_id) * 100.0 /
        COUNT(DISTINCT oi.order_id),
        2
    ) AS return_percentage
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
LEFT JOIN returns r
    ON oi.order_id = r.order_id
GROUP BY p.category
ORDER BY return_percentage DESC;""").show()

+--------------+-----------------+
|      category|return_percentage|
+--------------+-----------------+
|          Toys|            10.04|
|Home & Kitchen|            10.03|
|        Sports|            10.03|
|   Electronics|            10.02|
|         Books|            10.02|
|        Beauty|            10.02|
|      Clothing|             9.97|
+--------------+-----------------+



In [22]:
spark.sql("""WITH payment_counts AS (
    SELECT
        c.state,
        o.payment_mode,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (
            PARTITION BY c.state
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    GROUP BY
        c.state,
        o.payment_mode
)

SELECT
    state,
    payment_mode,
    cnt
FROM payment_counts
WHERE rn = 1
ORDER BY state;""").show()

+-----+----------------+-----+
|state|    payment_mode|  cnt|
+-----+----------------+-----+
|   CA|             UPI|20246|
|   FL|      Debit Card|20010|
|   GA|     Net Banking|20041|
|   IL|Cash on Delivery|20498|
|   MI|      Debit Card|20416|
|   NC|     Net Banking|19596|
|   NY|      Debit Card|20369|
|   OH|     Net Banking|20351|
|   TX|             UPI|20065|
|   WA|             UPI|20244|
+-----+----------------+-----+

